# RL Constrained Dataset DQN Training

This notebook:
1. Loads offline RL tensors from `data/processed/rl_tensors_*_constrained.npz`
2. Loads DQN hyperparameters from `configs/model.yaml`
3. Trains a dueling DQN using `src/rl/dqn.py::train_dqn`
4. Saves the trained model to `models/`
5. Runs offline evaluation using `src/ope/offline_eval.py`:
    - Mean Squared TD Error (MSTE)
    - Direct Q-based value estimate of greedy policy
    - Action agreement with logged (behavior) policy


## 1. Imports & Paths

In [1]:
import os
import sys
from pathlib import Path
import torch
from dataclasses import replace
from collections import OrderedDict

PROJECT_ROOT = Path(os.getcwd()).resolve().parent
sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /Users/ethanbobrik/Projects/MLB-Bullpen-Strategy


In [2]:
from src.rl.dqn import (
    load_dqn_training_config,
    train_dqn,
    BullpenOfflineDataset,
    RLDatasetConfig,
)
from src.ope.offline_eval import (
    evaluate_td_error_full_mse,
    direct_policy_value_estimate,
    compute_policy_behavior_stats,
    compute_q_distributions,
    summarize_policy_behavior_stats,
    summarize_q_distributions,
)

## 2. Configurations

In [3]:
DATA_DIR = PROJECT_ROOT / "data"
PROC_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

YEAR_TAG = "2022_2023"  # must match 01_build_dataset YEARS range
RL_TENSORS_PATH = PROC_DIR / f"rl_tensors_{YEAR_TAG}_constrained.npz"
MODEL_CFG_PATH = CONFIG_DIR / "model.yaml"
def outpath(model: str):
    MODEL_OUT_PATH = MODELS_DIR / f"{model}_dqn_bullpen_{YEAR_TAG}_constrained.pt"
    return MODEL_OUT_PATH

print("RL tensors:", RL_TENSORS_PATH)
print("Model config:", MODEL_CFG_PATH)

RL tensors: /Users/ethanbobrik/Projects/MLB-Bullpen-Strategy/data/processed/rl_tensors_2022_2023_constrained.npz
Model config: /Users/ethanbobrik/Projects/MLB-Bullpen-Strategy/configs/model.yaml


## 3. Load Dataset & Build Model

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

train_cfg = load_dqn_training_config(
    model_config_path=MODEL_CFG_PATH,
    data_path=RL_TENSORS_PATH,
    device=device,
)

train_cfg

Using device: cpu


DQNTrainingConfig(data_path=PosixPath('/Users/ethanbobrik/Projects/MLB-Bullpen-Strategy/data/processed/rl_tensors_2022_2023_constrained.npz'), device='cpu', batch_size=512, weight_decay=0.0, lr=0.001, gamma=0.99, max_steps=50000, target_update_interval=1000, log_interval=1000, val_fraction=0.2, early_stopping_patience=10, early_stopping_min_delta=0.01, hidden_size=258, num_layers=3, dropout=0.1, grad_clip_max_norm=0.0, yaml_num_actions=11)

In [5]:
# We treat train_cfg as a base config coming from configs/model.yaml
base_cfg = train_cfg

# 1) Shallow model: smaller net, less layers, light regularization
cfg_shallow = replace(
    base_cfg,
    hidden_size=128,          # fewer hidden units
    num_layers=2,             # shallower
    dropout=0.05,             # small dropout
    weight_decay=0.0,         # no L2
    grad_clip_max_norm=0.0,   # no grad clipping
)

# 2) Deeper model: larger network, same regularization as base
cfg_deep = replace(
    base_cfg,
    hidden_size=256,
    num_layers=4,
    dropout=0.10,
    weight_decay=0.0,
    grad_clip_max_norm=0.0,
)

# 3) Constrained model: like deep, but with weight decay + grad clipping
cfg_constrained = replace(
    base_cfg,
    hidden_size=256,
    num_layers=4,
    dropout=0.10,
    weight_decay=1e-4,        # L2 regularization
    grad_clip_max_norm=5.0,   # clip gradients by global norm
)

cfg_shallow, cfg_deep, cfg_constrained

(DQNTrainingConfig(data_path=PosixPath('/Users/ethanbobrik/Projects/MLB-Bullpen-Strategy/data/processed/rl_tensors_2022_2023_constrained.npz'), device='cpu', batch_size=512, weight_decay=0.0, lr=0.001, gamma=0.99, max_steps=50000, target_update_interval=1000, log_interval=1000, val_fraction=0.2, early_stopping_patience=10, early_stopping_min_delta=0.01, hidden_size=128, num_layers=2, dropout=0.05, grad_clip_max_norm=0.0, yaml_num_actions=11),
 DQNTrainingConfig(data_path=PosixPath('/Users/ethanbobrik/Projects/MLB-Bullpen-Strategy/data/processed/rl_tensors_2022_2023_constrained.npz'), device='cpu', batch_size=512, weight_decay=0.0, lr=0.001, gamma=0.99, max_steps=50000, target_update_interval=1000, log_interval=1000, val_fraction=0.2, early_stopping_patience=10, early_stopping_min_delta=0.01, hidden_size=256, num_layers=4, dropout=0.1, grad_clip_max_norm=0.0, yaml_num_actions=11),
 DQNTrainingConfig(data_path=PosixPath('/Users/ethanbobrik/Projects/MLB-Bullpen-Strategy/data/processed/rl_

In [6]:
ds = BullpenOfflineDataset(
    RLDatasetConfig(
        data_path=base_cfg.data_path,
        device=base_cfg.device,
    )
)

print("Dataset size:", len(ds))
print("State dim:", ds.state_dim)
print("Num actions:", ds.num_actions)
print("H (next hitters window):", ds.H)
print("R (max relievers per team):", ds.R)

Dataset size: 22249
State dim: 208
Num actions: 11
H (next hitters window): 5
R (max relievers per team): 10


## 4. Create Dueling DQN Model + Trainer

This calls `train_dqn(train_cfg)`, which:
 - loads `BullpenOfflineDataset` from `train_cfg.data_path`
 - splits into train/val by `train_cfg.val_fraction`
 - trains a dueling DQN with a target network
 - logs TD-error periodically using `evaluate_td_error` in `dqn.py`

This is done for 3 different models with differing complexity,
1. Shallow DQN Model
2. Deep DQN Model
3. Deep DQN Model with Gradient Clipping + Weight Decay

In [7]:
print("=== Training SHALLOW DQN model ===")
shallow_dqn = train_dqn(cfg_shallow)

=== Training SHALLOW DQN model ===
[DQN] step=0 loss=3121.16284
      val_td_error=2494.70118
      (new best val TD: 2494.70118)
[DQN] step=1000 loss=330.05817
      val_td_error=281.68612
      (new best val TD: 281.68612)
[DQN] step=2000 loss=194.45761
      val_td_error=251.89693
      (new best val TD: 251.89693)
[DQN] step=3000 loss=457.28821
      val_td_error=232.27659
      (new best val TD: 232.27659)
[DQN] step=4000 loss=446.10904
      val_td_error=207.70448
      (new best val TD: 207.70448)
[DQN] step=5000 loss=176.39262
      val_td_error=192.67871
      (new best val TD: 192.67871)
[DQN] step=6000 loss=145.21051
      val_td_error=168.45324
      (new best val TD: 168.45324)
[DQN] step=7000 loss=249.21677
      val_td_error=168.08952
      (new best val TD: 168.08952)
[DQN] step=8000 loss=154.35889
      val_td_error=154.63358
      (new best val TD: 154.63358)
[DQN] step=9000 loss=181.91653
      val_td_error=151.12009
      (new best val TD: 151.12009)
[DQN] step=1000

In [8]:
print("=== Training DEEP DQN model ===")
deep_dqn = train_dqn(cfg_deep)

=== Training DEEP DQN model ===
[DQN] step=0 loss=86.98798
      val_td_error=170.31623
      (new best val TD: 170.31623)
[DQN] step=1000 loss=59.65511
      val_td_error=19.35085
      (new best val TD: 19.35085)
[DQN] step=2000 loss=23.09974
      val_td_error=10.84133
      (new best val TD: 10.84133)
[DQN] step=3000 loss=16.35575
      val_td_error=6.19537
      (new best val TD: 6.19537)
[DQN] step=4000 loss=5.57462
      val_td_error=3.44068
      (new best val TD: 3.44068)
[DQN] step=5000 loss=2.19848
      val_td_error=2.15796
      (new best val TD: 2.15796)
[DQN] step=6000 loss=1.87118
      val_td_error=1.48104
      (new best val TD: 1.48104)
[DQN] step=7000 loss=0.93640
      val_td_error=1.06646
      (new best val TD: 1.06646)
[DQN] step=8000 loss=1.12560
      val_td_error=0.83058
      (new best val TD: 0.83058)
[DQN] step=9000 loss=0.84311
      val_td_error=0.70472
      (new best val TD: 0.70472)
[DQN] step=10000 loss=0.70364
      val_td_error=0.63649
      (new b

In [9]:
print("=== Training CONSTRAINED DQN model (weight decay + grad clipping) ===")
constrained_dqn = train_dqn(cfg_constrained)

=== Training CONSTRAINED DQN model (weight decay + grad clipping) ===
[DQN] step=0 loss=428.80542
      val_td_error=366.00220
      (new best val TD: 366.00220)
[DQN] step=1000 loss=68.25876
      val_td_error=29.26925
      (new best val TD: 29.26925)
[DQN] step=2000 loss=78.99712
      val_td_error=17.42465
      (new best val TD: 17.42465)
[DQN] step=3000 loss=20.73197
      val_td_error=11.49668
      (new best val TD: 11.49668)
[DQN] step=4000 loss=15.35073
      val_td_error=5.87453
      (new best val TD: 5.87453)
[DQN] step=5000 loss=4.32049
      val_td_error=3.56909
      (new best val TD: 3.56909)
[DQN] step=6000 loss=4.86650
      val_td_error=2.45237
      (new best val TD: 2.45237)
[DQN] step=7000 loss=2.30185
      val_td_error=1.73903
      (new best val TD: 1.73903)
[DQN] step=8000 loss=2.12817
      val_td_error=1.29240
      (new best val TD: 1.29240)
[DQN] step=9000 loss=0.86475
      val_td_error=1.06404
      (new best val TD: 1.06404)
[DQN] step=10000 loss=0.764

## 5. Save trained model weights

In [10]:
shallow_outpath = outpath('shallow')
deep_outpath = outpath('deep')
constrained_outpath = outpath('constrained')


torch.save(shallow_dqn.state_dict(), shallow_outpath)
torch.save(deep_dqn.state_dict(), deep_outpath)
torch.save(constrained_dqn.state_dict(), constrained_outpath)

## Offline Policy Evaluation (OPE)

Now we use `src/ope/offline_eval.py` to:
- load the saved model and dataset
- compute:
    - Mean Squared TD Error (MSTE)
    - Direct Q-based value of the greedy policy
    - Action agreement with the logged policy

In [11]:
eval_ds = BullpenOfflineDataset(
    RLDatasetConfig(
        data_path=RL_TENSORS_PATH,
        device=device,
    )
)
eval_loader = torch.utils.data.DataLoader(eval_ds, batch_size=2048, shuffle=False)

print("Eval dataset size:", len(eval_ds))
print("State dim:", eval_ds.state_dim)
print("Num actions:", eval_ds.num_actions)

Eval dataset size: 22249
State dim: 208
Num actions: 11


In [12]:
# Ensure models are on correct device
dqn_shallow = shallow_dqn.to(device)
dqn_deep = deep_dqn.to(device)
dqn_constrained = constrained_dqn.to(device)

gamma = cfg_constrained.gamma

models_to_eval = OrderedDict(
    [
        ("shallow", dqn_shallow),
        ("deep", dqn_deep),
        ("constrained", dqn_constrained),
    ]
)

all_results = {}

for name, model in models_to_eval.items():
    print(f"\n==================== {name.upper()} MODEL ====================")
    model.eval()

    # 1) TD Error (MSTE)
    mste_i = evaluate_td_error_full_mse(
        model=model,
        loader=eval_loader,
        gamma=gamma,
        device=device,
    )

    # 2) Direct Q-based value estimate
    dm_value_i = direct_policy_value_estimate(
        model=model,
        loader=eval_loader,
        device=device,
    )

    # 3) Policy vs behavior stats
    policy_stats_i = compute_policy_behavior_stats(
        model=model,
        loader=eval_loader,
        device=device,
    )

    # 4) Q-value distribution stats
    q_stats_i = compute_q_distributions(
        model=model,
        loader=eval_loader,
        device=device,
    )

    all_results[name] = {
        "mste": mste_i,
        "dm_value": dm_value_i,
        "policy_stats": policy_stats_i,
        "q_stats": q_stats_i,
    }

    agreement_rate_i = policy_stats_i["agreement_rate"]
    behavior_pull_rate_i = policy_stats_i["behavior_pull_rate"]
    policy_pull_rate_i = policy_stats_i["policy_pull_rate"]

    print(f"TD Error (MSTE):              {mste_i:.6f}")
    print(f"Direct Q-based V(pi_greedy):  {dm_value_i:.6f}")
    print(f"Action agreement rate:        {agreement_rate_i*100:.3f}%")
    print(f"Behavior pull rate:           {behavior_pull_rate_i*100:.3f}%")
    print(f"Policy pull rate:             {policy_pull_rate_i*100:.3f}%")

    print("\n[Policy vs Behavior stats]")
    summarize_policy_behavior_stats(policy_stats_i)

    print("\n[Q-value distribution stats]")
    summarize_q_distributions(q_stats_i)
    print("======================================================")


==================== SHALLOW MODEL ====================
TD Error (MSTE):              80.755520
Direct Q-based V(pi_greedy):  161.419794
Action agreement rate:        2.180%
Behavior pull rate:           20.226%
Policy pull rate:             99.721%

[Policy vs Behavior stats]
=== Policy vs Behavior Stats ===
Num samples:   22249
Num actions:   11

Behavior pull rate: 20.23%
Policy pull rate:   99.72%
Action agreement:   2.18%

Behavior action counts (per action index):
[17749   643   687   596   579   410   434   369   308   270   204]
Policy action counts (per action index):
[  62 2198  722 4282  377 7937 1360 2300 2663  288   60]
Valid action counts (per action index):
[22249 21922 21209 21561 21081 21469 20782 21433 20896 20575 20860]

[Q-value distribution stats]
=== Q Distribution Stats ===
q_all_valid: n=234037, mean=157.324, std=41.470, min=-20.538, max=340.457
q_stay: n=22249, mean=156.214, std=41.215, min=3.029, max=323.297
q_best_pull: n=22249, mean=161.417, std=42.215, min

In [13]:
import numpy as np
from pathlib import Path

npz = np.load(Path("../data/processed/rl_tensors_2022_2023.npz"))

for key in ["reward_folded"]:
    x = npz[key]
    print(key, "shape:", x.shape)
    print(
        key,
        "mean:", float(x.mean()),
        "std:", float(x.std()),
        "min:", float(x.min()),
        "max:", float(x.max()),
    )

reward_folded shape: (407660,)
reward_folded mean: -0.010023725219070911 std: 0.6859701871871948 min: -7.711379528045654 max: 1.1493159532546997
